# Eight-study scratch overfit GPU validation

This private-input notebook first runs a synthetic CUDA forward/backward/optimizer probe, then optionally executes the fixed fold-0, seed-42, eight-study train-subset run. It does not install packages, download data, inspect reports, or print study identifiers. The default is disabled. Enable only after reviewing the package and receiving explicit execution authorization.

Outputs and the state-dict checkpoint are written only to a new private working-directory path. The bounded run is an engineering smoke test, not validation or generalization evidence.

In [ ]:
import json
import sys
import time
from pathlib import Path

RUN_TRAINING = False
RUN_GPU_PROBE = False
INPUT_ROOT = Path('/kaggle/input/datasets/davidsulleiro/rsna-knee-five-study-reader-private')
IMAGE_ROOT = Path('/kaggle/input/competitions/rsna-knee-abnormality-detection/train_series')
OUTPUT_DIR = Path('/kaggle/working/kaggle-eight-study-overfit/run01')
CONFIG_PATH = INPUT_ROOT / 'configs/kaggle_eight_study_overfit.json'

def resolve_unique_input(filename):
    matches = []
    for candidate in (INPUT_ROOT / 'data/private' / filename, INPUT_ROOT / filename):
        if candidate.is_file():
            matches.append(candidate.resolve(strict=True))
    if len(set(matches)) != 1:
        raise RuntimeError('private_input_resolution_failed')
    return matches[0]

if not INPUT_ROOT.is_dir() or not IMAGE_ROOT.is_dir():
    raise RuntimeError('observed_input_mount_unavailable')
if not CONFIG_PATH.is_file():
    raise RuntimeError('training_config_unavailable')
config_value = json.loads(CONFIG_PATH.read_text(encoding='utf-8'))
manifest_path = resolve_unique_input('500-study-training-cohort.json')
folds_path = resolve_unique_input('500-study-training-folds.json')
config_value.update({
    'manifest_path': str(manifest_path),
    'folds_path': str(folds_path),
    'image_root': str(IMAGE_ROOT),
    'output_dir': str(OUTPUT_DIR),
})
sys.path.insert(0, str(INPUT_ROOT / 'src'))
from rsna_knee.training.baseline import TrainingConfig, run_training
config = TrainingConfig.from_mapping(config_value)
print({'mode': config.mode, 'fold': config.fold, 'seed': config.seed, 'device': config.device, 'training_enabled': RUN_TRAINING, 'gpu_probe_enabled': RUN_GPU_PROBE, 'private_input_update_pending': False})

In [ ]:
import torch

def run_cuda_numerical_probe():
    if not torch.cuda.is_available():
        raise RuntimeError('cuda_unavailable')
    device = torch.device('cuda')
    torch.manual_seed(42)
    torch.cuda.manual_seed_all(42)
    torch.use_deterministic_algorithms(True)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False
    torch.backends.cuda.matmul.allow_tf32 = False
    model = torch.nn.Linear(16, 12).to(device)
    optimizer = torch.optim.SGD(model.parameters(), lr=0.001)
    inputs = torch.ones((2, 16), device=device)
    targets = torch.zeros((2, 12), device=device)
    optimizer.zero_grad(set_to_none=True)
    loss = torch.nn.functional.binary_cross_entropy_with_logits(model(inputs), targets)
    loss.backward()
    optimizer.step()
    torch.cuda.synchronize(device)
    if not torch.isfinite(loss):
        raise RuntimeError('gpu_probe_nonfinite')
    return {'status': 'passed', 'cuda_device_count': torch.cuda.device_count(), 'loss_finite': True, 'peak_memory_bytes': int(torch.cuda.max_memory_allocated(device))}

if RUN_GPU_PROBE:
    gpu_probe = run_cuda_numerical_probe()
    print({'gpu_probe': gpu_probe})
else:
    print({'gpu_probe': 'disabled'})

In [ ]:
if RUN_TRAINING:
    if OUTPUT_DIR.exists():
        raise RuntimeError('private_output_must_be_new')
    started = time.perf_counter()
    result = run_training(config, execute=True)
    safe_summary = {key: result.get(key) for key in ('status', 'mode', 'fold', 'training_study_count', 'steps_completed', 'train_loss_start', 'train_loss_end', 'runtime', 'interpretation_limit')}
    safe_summary['notebook_elapsed_seconds'] = time.perf_counter() - started
    print({'training': safe_summary})
else:
    print({'training': 'disabled'})